In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, ConfusionMatrixDisplay
)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import joblib, os, warnings
warnings.filterwarnings('ignore')
print('Imports OK')

Imports OK


In [3]:
CSV_PATH = '/home/chinmaya/Desktop/coding/aiagent/health_care/data sets/dibetis/diabetes.csv'  # update to your path
df = pd.read_csv(CSV_PATH)
print(f'Shape: {df.shape}')
df.head()

Shape: (768, 9)


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [4]:
# ── BUG 6 FIX: Remove broken indentation ─────────────────────
# Original code had 4-space indentation at module level with no
# enclosing function or if block — Python raises IndentationError.
# These lines run at module level, so NO indentation.

df = df.copy()

# Columns where 0 is biologically impossible — replace with NaN
impossible_zero_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
for col in impossible_zero_cols:
    df[col] = df[col].replace(0, np.nan)

print('Missing values after replacing impossible zeros:')
print(df.isnull().sum()[df.isnull().sum() > 0])

Missing values after replacing impossible zeros:
Glucose            5
BloodPressure     35
SkinThickness    227
Insulin          374
BMI               11
dtype: int64


In [5]:
X = df.drop('Outcome', axis=1)
y = df['Outcome']

feature_names = X.columns.tolist()
print(f'Features: {feature_names}')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {X_train.shape}  Test: {X_test.shape}')

Features: ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']
Train: (614, 8)  Test: (154, 8)


In [6]:
pipelines = {
    'Logistic Regression': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler',  StandardScaler()),
        ('model',   LogisticRegression(max_iter=1000, random_state=42))
    ]),
    'Random Forest': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler',  StandardScaler()),
        ('model',   RandomForestClassifier(
            n_estimators=200, max_depth=8, min_samples_leaf=5,
            random_state=42, class_weight='balanced'
        ))
    ]),
    'Gradient Boosting': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler',  StandardScaler()),
        ('model',   GradientBoostingClassifier(
            n_estimators=200, learning_rate=0.05, max_depth=4, random_state=42
        ))
    ])
}
print('Pipelines built')

Pipelines built


In [7]:
# ── BUG 7 FIX: define function BEFORE calling it ──────────────
# Original Cell 7 had: results = train_and_evaluate(...)  ← call first
#                       def plot_results(...): ...         ← then define
# This works by accident in Python (defs execute sequentially),
# but it is confusing and breaks the rule: define before you call.

def train_and_evaluate(pipelines, X_train, X_test, y_train, y_test):
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    results = {}

    print('=' * 55)
    print('MODEL TRAINING AND EVALUATION')
    print('=' * 55)

    for name, pipe in pipelines.items():
        print(f'\nTraining: {name}')
        cv_scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='roc_auc')
        print(f'  CV ROC-AUC: {cv_scores.mean():.3f} +/- {cv_scores.std():.3f}')

        pipe.fit(X_train, y_train)
        y_pred      = pipe.predict(X_test)
        y_pred_prob = pipe.predict_proba(X_test)[:, 1]
        auc         = roc_auc_score(y_test, y_pred_prob)

        print(f'  Test ROC-AUC: {auc:.3f}')
        print(classification_report(y_test, y_pred, target_names=['No Diabetes', 'Diabetes']))

        results[name] = {
            'pipeline':    pipe,
            'auc':         auc,
            'cv_mean':     cv_scores.mean(),
            'cv_std':      cv_scores.std(),
            'y_pred':      y_pred,
            'y_pred_prob': y_pred_prob
        }
    return results

print('train_and_evaluate defined')

train_and_evaluate defined


In [8]:
def plot_results(results, y_test, feature_names):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle('Diabetes Model Evaluation', fontsize=14, fontweight='bold')

    ax = axes[0]
    for name, res in results.items():
        fpr, tpr, _ = roc_curve(y_test, res['y_pred_prob'])
        ax.plot(fpr, tpr, label=f"{name} (AUC={res['auc']:.3f})")
    ax.plot([0,1],[0,1],'k--', alpha=0.4, label='Random')
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title('ROC Curves')
    ax.legend()

    best_name = max(results, key=lambda k: results[k]['auc'])
    best      = results[best_name]
    ConfusionMatrixDisplay(
        confusion_matrix(y_test, best['y_pred']),
        display_labels=['No Diabetes', 'Diabetes']
    ).plot(ax=axes[1], colorbar=False)
    axes[1].set_title(f'Confusion Matrix: {best_name}')

    ax  = axes[2]
    mdl = best['pipeline'].named_steps['model']
    if hasattr(mdl, 'feature_importances_'):
        importance = mdl.feature_importances_
        idx = np.argsort(importance)
        ax.barh(np.array(feature_names)[idx], importance[idx], color='steelblue')
        ax.set_title(f'Feature Importance: {best_name}')
        ax.set_xlabel('Importance')
    else:
        coef = np.abs(mdl.coef_[0])
        idx  = np.argsort(coef)
        ax.barh(np.array(feature_names)[idx], coef[idx], color='steelblue')
        ax.set_title(f'Coefficient Magnitude: {best_name}')

    plt.tight_layout()
    plt.savefig('diabetes_evaluation.png', dpi=100)
    plt.show()

print('plot_results defined')

plot_results defined


In [9]:
# Now call both functions — defined above, called here
results = train_and_evaluate(pipelines, X_train, X_test, y_train, y_test)
plot_results(results, y_test, feature_names)

MODEL TRAINING AND EVALUATION

Training: Logistic Regression
  CV ROC-AUC: 0.843 +/- 0.019
  Test ROC-AUC: 0.813
              precision    recall  f1-score   support

 No Diabetes       0.75      0.82      0.78       100
    Diabetes       0.60      0.50      0.55        54

    accuracy                           0.71       154
   macro avg       0.68      0.66      0.67       154
weighted avg       0.70      0.71      0.70       154


Training: Random Forest
  CV ROC-AUC: 0.837 +/- 0.021
  Test ROC-AUC: 0.819
              precision    recall  f1-score   support

 No Diabetes       0.83      0.75      0.79       100
    Diabetes       0.61      0.72      0.66        54

    accuracy                           0.74       154
   macro avg       0.72      0.74      0.73       154
weighted avg       0.75      0.74      0.74       154


Training: Gradient Boosting
  CV ROC-AUC: 0.808 +/- 0.020
  Test ROC-AUC: 0.811
              precision    recall  f1-score   support

 No Diabetes       0

In [10]:
def save_best_model(results):
    best_name = max(results, key=lambda k: results[k]['auc'])
    best_pipe = results[best_name]['pipeline']
    os.makedirs('models', exist_ok=True)
    joblib.dump(best_pipe, 'models/diabetes_model.pkl')
    print(f'Best model: {best_name}  (AUC = {results[best_name]["auc"]:.3f})')
    print('Saved: models/diabetes_model.pkl')
    return best_name, best_pipe

best_name, best_pipe = save_best_model(results)

Best model: Random Forest  (AUC = 0.819)
Saved: models/diabetes_model.pkl


In [11]:
# ── BUG 8 FIX: predict_diabetes() was completely missing ──────
# The aggregator calls this function at runtime.
# Without it, the diabetes model cannot be used in the CDSS pipeline.

DIABETES_FEATURES = [
    'Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness',
    'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age'
]

def predict_diabetes(mdl, patient_data: dict) -> dict:
    defaults = {
        'Pregnancies': 0, 'Glucose': 100, 'BloodPressure': 72,
        'SkinThickness': 20, 'Insulin': 80, 'BMI': 25.0,
        'DiabetesPedigreeFunction': 0.5, 'Age': 35
    }
    row = {f: patient_data.get(f, defaults[f]) for f in DIABETES_FEATURES}
    X   = pd.DataFrame([row])[DIABETES_FEATURES]

    prob = float(mdl.predict_proba(X)[0][1])
    risk = 'High' if prob >= 0.70 else 'Moderate' if prob >= 0.40 else 'Low'

    return {
        'disease':     'Diabetes',
        'probability': round(prob, 3),
        'risk_level':  risk,
        'confidence':  f'{prob * 100:.1f}%'
    }

print('predict_diabetes defined')

predict_diabetes defined


In [12]:
high_risk = {
    'Pregnancies': 6, 'Glucose': 148, 'BloodPressure': 72,
    'SkinThickness': 35, 'Insulin': 0, 'BMI': 33.6,
    'DiabetesPedigreeFunction': 0.627, 'Age': 50
}
low_risk = {
    'Pregnancies': 1, 'Glucose': 85, 'BloodPressure': 66,
    'SkinThickness': 29, 'Insulin': 0, 'BMI': 26.6,
    'DiabetesPedigreeFunction': 0.351, 'Age': 31
}

print('High risk patient:', predict_diabetes(best_pipe, high_risk))
print('Low risk patient: ', predict_diabetes(best_pipe, low_risk))
print('\nDISCLAIMER: This is a predictive model only, not a medical diagnosis.')

High risk patient: {'disease': 'Diabetes', 'probability': 0.504, 'risk_level': 'Moderate', 'confidence': '50.4%'}
Low risk patient:  {'disease': 'Diabetes', 'probability': 0.052, 'risk_level': 'Low', 'confidence': '5.2%'}

DISCLAIMER: This is a predictive model only, not a medical diagnosis.
